# Parser da Tabela de Anotações

Transforma a planilha `Tabela de itens - Arqgen - Pavimento completo.xlsx` numa tabela única, para ser cruzada com o DXF `003_complete_floor` no próximo notebook.

A planilha tem uma aba por ambiente. O `Block Name` é a chave de junção com os INSERTs do DXF, e aparece em duas formas:

- Com o prefixo de xref, como no desenho original
- Com o prefixo `ARQGEN_`, para blocos que quem anotou criou numa cópia própria do desenho

A saída é uma linha por item x block name, porque é nesse grão que a comparação com os INSERTs do desenho acontece.

## 1. Setup

In [1]:
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Não foi possível localizar a raiz do repositório.")

REPO_ROOT = find_repo_root(Path.cwd())
DADOS = REPO_ROOT / "data" / "labeled_data"
PLANILHA = DADOS / "Tabela de itens - Arqgen - Pavimento completo.xlsx"
SAIDA = DADOS / "anotacoes.csv"

if not PLANILHA.exists():
    raise FileNotFoundError(f"Planilha não encontrada: {PLANILHA}")

print("Fonte:", PLANILHA.name)

Fonte: Tabela de itens - Arqgen - Pavimento completo.xlsx


## 2. Parse

Uma regra por tipo de linha, e só as que a planilha realmente contém:

- Linha em branco é ignorada
- Linha com apenas `Categoria` é um subtítulo dentro da aba, e passa a valer para as linhas seguintes
- linha com `Item` e `Block Name` é um item
- Qualquer outra coisa vira anomalia

In [2]:
COLUNAS = ["Categoria", "Item", "Block Name", "Quantidade"]
JUNCAO = re.compile(r"\s+(ou|e)\s+", re.IGNORECASE)

def texto(valor: object) -> str | None:
    if valor is None or pd.isna(valor):
        return None
    limpo = str(valor).strip()
    return limpo or None

def inteiro(valor: object) -> int | None:
    try:
        numero = float(valor)
    except (TypeError, ValueError):
        return None
    return int(numero) if numero.is_integer() and numero >= 0 else None

def parse_planilha(caminho: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    excel = pd.ExcelFile(caminho, engine="openpyxl")
    itens: list[dict] = []
    anomalias: list[dict] = []

    for aba in excel.sheet_names:
        folha = pd.read_excel(excel, sheet_name=aba, header=None, dtype=object)
        # A quinta coluna (observações) não existe em toda aba.
        while folha.shape[1] < 5:
            folha[folha.shape[1]] = None

        registros = folha.iloc[:, :5].values.tolist()
        cabecalho = [texto(valor) for valor in registros[0][:4]]
        if cabecalho != COLUNAS:
            raise ValueError(f"{aba}: cabeçalho inesperado {cabecalho}")

        ambiente = aba.strip()
        secao: str | None = None

        for linha, registro in enumerate(registros[1:], start=2):
            categoria, item, spec, observacao = (
                texto(registro[i]) for i in (0, 1, 2, 4)
            )
            quantidade = inteiro(registro[3])

            if not any([categoria, item, spec, observacao]) and quantidade is None:
                continue

            if categoria and not item and not spec:
                secao = categoria
                continue

            if not item or not spec:
                anomalias.append({
                    "ambiente": ambiente, "linha": linha, "item": item,
                    "problema": "linha sem Item ou sem Block Name",
                })
                continue

            if not categoria:
                anomalias.append({
                    "ambiente": ambiente, "linha": linha, "item": item,
                    "problema": "linha sem Categoria",
                })

            partes = JUNCAO.split(spec)
            nomes = [parte.strip() for parte in partes[0::2] if parte.strip()]
            operadores = {operador.lower() for operador in partes[1::2]}

            if quantidade is None:
                anomalias.append({
                    "ambiente": ambiente, "linha": linha, "item": item,
                    "problema": f"quantidade inválida: {registro[3]!r}",
                })
            if len(nomes) != len(set(nomes)):
                anomalias.append({
                    "ambiente": ambiente, "linha": linha, "item": item,
                    "problema": "mesmo block name repetido na linha",
                })

            for nome in nomes:
                # A planilha usa "--" para marcar bloco que falta no desenho.
                if not any(caractere.isalnum() for caractere in nome):
                    anomalias.append({
                        "ambiente": ambiente, "linha": linha, "item": item,
                        "problema": f"block name sem conteúdo: {nome!r}",
                    })
                itens.append({
                    "ambiente": ambiente,
                    "secao": secao,
                    "categoria": categoria,
                    "item": item,
                    "block_name": nome,
                    "quantidade": quantidade,
                    "juncao": "/".join(sorted(operadores)) or None,
                    "observacao": observacao,
                    "linha": linha,
                })

    anotacoes = pd.DataFrame(itens)
    # Int64 aceita nulo sem virar float: quantidade sai 14 e não 14.0, no CSV também.
    anotacoes["quantidade"] = anotacoes["quantidade"].astype("Int64")
    return anotacoes, pd.DataFrame(anomalias)

## 3. Resultado

In [3]:
anotacoes, anomalias = parse_planilha(PLANILHA)

resumo = pd.DataFrame([{
    "ambientes": anotacoes["ambiente"].nunique(),
    "linhas_de_item": len(anotacoes.drop_duplicates(["ambiente", "linha"])),
    "block_names": len(anotacoes),
    "block_names_distintos": anotacoes["block_name"].nunique(),
    "distintos_arqgen": anotacoes.loc[
        anotacoes["block_name"].str.startswith("ARQGEN_"), "block_name"
    ].nunique(),
    "com_observacao": int(anotacoes["observacao"].notna().sum()),
    "com_juncao": int(anotacoes["juncao"].notna().sum()),
    "anomalias": len(anomalias),
}])
display(resumo)

display(anotacoes.head(10))

,ambientes,linhas_de_item,block_names,block_names_distintos,distintos_arqgen,com_observacao,com_juncao,anomalias
0,31,324,343,151,85,24,26,13


,ambiente,secao,categoria,item,block_name,quantidade,juncao,observacao,linha
0,Posto para Anestesia e RPA,NaN,Mobiliário móvel,Lixo Comum,ARQGEN_LIXOCOMUM,14,NaN,NaN,2
1,Posto para Anestesia e RPA,NaN,Equipamento,Monitor,M095-21-PBAQ-0001-P-2S-01$0$EQL_MD_monitor lcd - 32_ - c...,14,NaN,NaN,3
2,Posto para Anestesia e RPA,NaN,Equipamento,Tablet,ARQGEN_TABLETMENOR,14,NaN,NaN,4
3,Posto para Anestesia e RPA,NaN,Mobiliário fixo,Régua de gases,M095-21-PBAQ-0001-P-2S-01$0$E071a-p,14,NaN,NaN,5
4,Posto para Anestesia e RPA,NaN,Mobiliário móvel,Maca,M095-21-PBAQ-0001-P-2S-01$0$E018b-p,14,NaN,NaN,6
5,Posto para Anestesia e RPA,NaN,Mobiliário móvel,Mesa lateral,M095-21-PBAQ-0001-P-2S-01$0$MOB 7499C,14,NaN,NaN,7
6,Posto para Anestesia e RPA,NaN,Mobiliário móvel,Cadeira de espera,M095-21-PBAQ-0001-P-2S-01$0$CADERIA TANTI - RSDESIGN,14,NaN,NaN,8
7,Posto para Anestesia e RPA,NaN,Mobiliário fixo,Bancada do Posto de Anestesia 01,ARQGEN_BANCADA_PANESTESIA,1,NaN,NaN,9
8,Posto para Anestesia e RPA,NaN,Equipamento,Computador,M095-21-PBAQ-0001-P-2S-01$0$E054a-p,5,NaN,NaN,10
9,Posto para Anestesia e RPA,NaN,Acessório,Chamada Enfermeira,M095-21-PBAQ-0001-P-2S-01$0$Chamada_enfermeira_110_pl,28,NaN,NaN,11


## 4. Anomalias

O que a planilha não formaliza fica visível aqui, com a aba e a linha de origem.

In [4]:
if anomalias.empty:
    print("Nenhuma anomalia.")
else:
    display(anomalias.sort_values(["ambiente", "linha"]).reset_index(drop=True))

,ambiente,linha,item,problema
0,Antecâmera,8,Nichos,linha sem Categoria
1,Biblioteca,2,Bancada atendimento,mesmo block name repetido na linha
2,Café,44,Armário Alto 55x90cm,linha sem Categoria
3,Café,44,Armário Alto 55x90cm,quantidade inválida: nan
4,Copa,45,Armário Alto 55x90cm,linha sem Categoria
5,Copa,45,Armário Alto 55x90cm,quantidade inválida: nan
6,Lavagem,8,Armário inox,block name sem conteúdo: '--'
7,Sala Leiliane,33,Armário Alto 55x90cm,linha sem Categoria
8,Sala Leiliane,33,Armário Alto 55x90cm,quantidade inválida: nan
9,Sala Maris,34,Armário Alto 55x90cm,linha sem Categoria


## 5. Exportar

É exportado um CSV ao lado da planilha de origem. É o que o próximo notebook carrega, sem precisar reabrir o `.xlsx` nem depender do `openpyxl`. Para cruzar com o DXF, a junção é `anotacoes["block_name"]` contra o nome de bloco dos INSERTs do `003_complete_floor`. Deve-se verificar como tratar `ARQGEN_*`.

In [5]:
anotacoes.to_csv(SAIDA, index=False)
print("Exportado:", SAIDA.relative_to(REPO_ROOT), f"({len(anotacoes)} linhas)")

Exportado: data/labeled_data/anotacoes.csv (343 linhas)
